# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** Logistic Regression first, then Random Forest for comparison.

**Why it fits:** The question is yes/no with an observed label (`is_declining_label`), which
the toolkit maps directly to "Logistic Regression, then Random Forest — readable → stronger."
My baseline is a ranking rule (a score used to prioritize a review queue), so the honest
comparison metric is precision@K, not accuracy — of the top K pages either method flags,
how many are actually declining? I start with Logistic Regression because its coefficients are
directly readable (same spirit as the baseline's transparent rule), then check whether Random
Forest earns its added complexity by actually beating it at the K values that matter, per the
skill's warning not to reward complexity alone.

In [1]:
!git clone https://github.com/Prakritibhandari07/FlyRank-ml-internship.git
%cd FlyRank-ml-internship

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Rows: {len(df)}, base rate: {df['is_declining_label'].mean():.3f}")

Cloning into 'FlyRank-ml-internship'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 154 (delta 62), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1.87 MiB | 10.03 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/FlyRank-ml-internship
Rows: 30000, base rate: 0.542


## 2. Split design

**Split:** grouped by `client_id`, 75/25 train/test, `GroupShuffleSplit` with a fixed seed.

**Why this is honest:** there are only 32 clients, and rows from the same client share
publishing batches (my baseline's top-20 review found 20/20 flagged rows came from just 2
clients with an identical `days_since_last_update = 104` — a shared content push). A random
row-level split would let the same client's near-duplicate pages appear in both train and test,
letting the model "memorize" a client's pattern rather than generalize to a new one. Grouping by
client_id means every client's rows land entirely in train or entirely in test — the model is
evaluated on clients it has never seen, which is the honest question ("does this generalize?").

In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients")
print(f"Test:  {len(test_df)} rows, {test_df['client_id'].nunique()} clients")
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Client overlap between train/test: {len(overlap)} (should be 0)")

Train: 22885 rows, 24 clients
Test:  7115 rows, 8 clients
Client overlap between train/test: 0 (should be 0)


## 3. Train + compare vs my baseline

Same split, same metric (precision@K), same label as the baseline. The baseline's score is
recomputed identically to Week-4, then both are ranked and evaluated ONLY on the test set —
the model never gets an advantage from being scored on rows it trained on.

Features exclude every label-derived and future-window column the baseline skill flagged as
forbidden: `trend_direction`, `trend_pct`, `is_declining_label`, `impressions_last_30d`,
`impressions_prev_30d`. IDs (`content_id`, `client_id`) are used for grouping only, never as
features.

In [7]:
# --- Recreate the Week-4 baseline rule, identically ------------------------------------
def baseline_score(frame):
    stale = frame["freshness_tier"].isin(["91-180", "181+"]).astype(int)
    measurable = ((frame["impressions_90d"] >= 100) & (frame["sessions_90d"] > 0)).astype(int)
    tier_median_ctr = frame.groupby("position_tier", observed=True)["ctr"].transform("median")
    ctr_underperform = np.where(measurable == 1, (frame["ctr"] < 0.7 * tier_median_ctr).astype(int), 0)
    return stale * measurable * ctr_underperform * frame["impressions_90d"]

df["baseline_score"] = baseline_score(df)
test_df = df.iloc[test_idx].copy()  # refresh with baseline_score attached

# --- Build model features (no leakage, missingness handled honestly) -------------------
FORBIDDEN = ["trend_direction", "trend_pct", "is_declining_label",
             "impressions_last_30d", "impressions_prev_30d",
             "content_id", "client_id", "baseline_score"]

numeric_features = [
    "days_since_last_update", "content_age_days", "impressions_90d", "clicks_90d",
    "pageviews_90d", "sessions_90d", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
    "word_count", "char_count"
]
categorical_features = ["position_tier", "freshness_tier", "content_type", "main_intent",
                         "competition_level", "age_tier"]

work = df.copy()
# Missingness is systematic by content_type (per data dictionary) — add has_ flags before filling,
# so a blind fillna(0) doesn't silently encode content_type into the features.
for col in ["search_volume", "competition", "cpc", "word_count", "char_count"]:
    work[f"has_{col}"] = work[col].notna().astype(int)
    work[col] = work[col].fillna(0)
for col in categorical_features:
    work[col] = work[col].fillna("unknown")

feature_cols = numeric_features + [f"has_{c}" for c in ["search_volume","competition","cpc","word_count","char_count"]]
X = pd.get_dummies(work[feature_cols + categorical_features], columns=categorical_features, drop_first=True)
y = work["is_declining_label"]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Train models ------------------------------------------------------------------------
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
logreg.fit(X_train_scaled, y_train)
logreg_scores = logreg.predict_proba(X_test_scaled)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

baseline_test_scores = test_df["baseline_score"].values
y_test_arr = y_test.values

# --- precision@K, same split, same metric, base rate included --------------------------
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate_test = y_test_arr.mean()
Ks = [20, 50, 100, 500]

comparison = pd.DataFrame({
    "K": Ks,
    "base_rate": [base_rate_test] * len(Ks),
    "baseline_precision@K": [precision_at_k(baseline_test_scores, y_test_arr, k) for k in Ks],
    "logreg_precision@K":   [precision_at_k(logreg_scores, y_test_arr, k) for k in Ks],
    "random_forest_precision@K": [precision_at_k(rf_scores, y_test_arr, k) for k in Ks],
})
print(f"Test set base rate: {base_rate_test:.3f} (n={len(y_test_arr)})\n")
comparison

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 4. Errors and interpretation

**Top features:** [name the actual top 2-3 features from each importance list — do the
logistic regression and random forest agree on what matters? Does the top feature make sense,
or is it "suspiciously perfect" per the skill's leakage warning?]

**Where the model is most wrong:** [summarize the false-positive breakdown by position_tier /
content_type — is there a pattern, e.g. does it over-flag a specific content_type or
position_tier the same way the baseline's page_3_5 weakness did?]

**3 concrete wrong cases:** [for each of the 3 rows in wrong_cases, one sentence: what the
model saw that made it flag this page, and why it was actually wrong]

In [6]:
# --- What does each model lean on? ------------------------------------------------------
logreg_importance = pd.Series(np.abs(logreg.coef_[0]), index=X.columns).sort_values(ascending=False)
print("Top 10 features — Logistic Regression (|standardized coefficient|):")
print(logreg_importance.head(10))

rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 10 features — Random Forest (impurity importance):")
print(rf_importance.head(10))

# Permutation importance — a more honest check than impurity importance alone (per skill)
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1)
perm_importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("\nTop 10 features — Random Forest (permutation importance):")
print(perm_importance.head(10))

# --- Where is the best model most wrong? Look at top-K false positives -----------------
best_scores = rf_scores if comparison["random_forest_precision@K"].sum() >= comparison["logreg_precision@K"].sum() else logreg_scores
best_name = "Random Forest" if best_scores is rf_scores else "Logistic Regression"

test_df = test_df.reset_index(drop=True)
test_df["model_score"] = best_scores
test_df["actual_declining"] = y_test_arr

top50 = test_df.sort_values("model_score", ascending=False).head(50)
false_positives = top50[top50["actual_declining"] == 0]
print(f"\nUsing {best_name} — of top 50 flagged, {len(false_positives)} were NOT actually declining")

print("\nFalse positives by position_tier:")
print(false_positives["position_tier"].value_counts())
print("\nFalse positives by content_type:")
print(false_positives["content_type"].value_counts())

# 3 concrete wrong cases
wrong_cases = false_positives.head(3)[["content_id", "model_score", "position_tier",
                                         "freshness_tier", "ctr", "impressions_90d", "content_type"]]
print("\n3 concrete wrong cases:")
wrong_cases

AttributeError: 'LogisticRegression' object has no attribute 'coef_'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.